# 01. Real-World Data Wrangling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week12/01.Real-World-Data-Wrangling/notebooks/01_01.Real-World-Data-Wrangling.ipynb)

## Learning Objectives
- Audit raw datasets for common real-world data defects (whitespace, mixed casing, formatted currency, heterogeneous dates).
- Clean and standardise text data using vectorised `.str` methods.
- Parse mixed date strings using `pd.to_datetime(..., format='mixed')`.
- Extract numeric floats from messy currency strings via regular expressions.
- Impute missing records and perform auxiliary lookups with `pd.merge()`.


## 1. Creating and Auditing Raw Ingestion Data
Real-world data collected from online forms, legacy CRM portals, and manual spreadsheets contains typos, varied date conventions, and formatted currency symbols.
Let's create a realistic student admissions dataset containing these defects.

In [ ]:
import pandas as pd
import numpy as np

# Raw dirty admissions table
dirty_admissions = pd.DataFrame({
    'applicant_id': ['ACU-0101', 'ACU-0102', 'ACU-0103', 'ACU-0104', 'ACU-0105', 'ACU-0106'],
    'full_name': ['  Liam NGUYEN  ', 'Emma Watson', 'oliver brown ', '  Sophia VU', 'Noah Taylor', 'Ava Wilson  '],
    'campus': ['syd', 'Melbourne', '  SYDNEY  ', 'bne', '  melb  ', 'Brisbane'],
    'applied_date': ['2026-01-15', '18/01/2026', '2026/01/22', '25-01-2026', '2026.02.01', '05/02/2026'],
    'enrolment_deposit': ['$1,500.00 AUD', '$1500', ' $2,250.50 ', '1500.00', '$0.00 AUD', ' $1,800.00 '],
    'scholarship_award': ['VC Award', None, 'STEM Grant', None, None, 'Alumni Grant'],
    'residency': ['Domestic', 'International', None, 'Domestic', 'International', 'Domestic']
})

# Campus subsidy lookup table
campus_subsidy_lookup = pd.DataFrame({
    'standard_campus': ['North Sydney', 'Melbourne', 'Brisbane'],
    'subsidy_rate': [0.15, 0.12, 0.18]
})

print('--- Raw Admissions Data ---')
display(dirty_admissions)

print('\nMissing values audit:')
print(dirty_admissions.isna().sum())

print('\nColumn data types:')
print(dirty_admissions.dtypes)

## 2. Text and Categorical Sanitisation
Notice the leading/trailing spaces in `full_name` and inconsistent campus labels (`'syd'`, `'SYDNEY'`, `'bne'`, `'melb'`).
We use `.str.strip()`, `.str.title()`, and a mapping dictionary to standardise these categories.

In [ ]:
cleaned = dirty_admissions.copy()

# 1. Clean names: strip surrounding whitespace and convert to Proper Case
cleaned['full_name'] = cleaned['full_name'].str.strip().str.title()

# 2. Standardise campus codes to official campus names
campus_mapping = {
    'syd': 'North Sydney',
    'sydney': 'North Sydney',
    'melb': 'Melbourne',
    'melbourne': 'Melbourne',
    'bne': 'Brisbane',
    'brisbane': 'Brisbane'
}
cleaned['campus'] = cleaned['campus'].str.strip().str.lower().map(campus_mapping)

display(cleaned[['applicant_id', 'full_name', 'campus']])

## 3. Financial Currency Cleaning and Mixed Date Parsing
The `enrolment_deposit` column currently contains strings with `$` symbols, commas, and currency codes (`AUD`).
The `applied_date` column has a mixture of ISO format (`YYYY-MM-DD`), Australian slash format (`DD/MM/YYYY`), and dot-delimited formats.
We use regex extraction and `format='mixed'` to transform them into numeric floats and `datetime64[ns]`.

In [ ]:
# Clean enrolment deposit to float
cleaned['enrolment_deposit'] = (
    cleaned['enrolment_deposit']
    .str.replace(r'[$,AUD\s]', '', regex=True)
    .astype(float)
)

# Standardise mixed date formats to pandas Timestamp
cleaned['applied_date'] = pd.to_datetime(cleaned['applied_date'], format='mixed')

# Impute missing scholarship and residency values
cleaned['scholarship_award'] = cleaned['scholarship_award'].fillna('No Scholarship')
cleaned['residency'] = cleaned['residency'].fillna('Domestic')

print('Cleaned Financial and Datetime Features:')
display(cleaned[['applicant_id', 'applied_date', 'enrolment_deposit', 'scholarship_award', 'residency']])

print(f'Deposit dtype: {cleaned["enrolment_deposit"].dtype}')
print(f'Applied Date dtype: {cleaned["applied_date"].dtype}')

## 4. Auxiliary Table Merging and Feature Engineering
Now that the campus names are standardised (`'North Sydney'`, `'Melbourne'`, `'Brisbane'`), we can safely join the campus subsidy reference table via a left merge.

In [ ]:
# Left join campus subsidy rates
enriched = pd.merge(
    cleaned,
    campus_subsidy_lookup,
    left_on='campus',
    right_on='standard_campus',
    how='left'
).drop(columns=['standard_campus'])

# Feature Engineering: calculate subsidy discount and net deposit payable
enriched['subsidy_kaud'] = (enriched['enrolment_deposit'] * enriched['subsidy_rate']).round(2)
enriched['net_deposit'] = (enriched['enrolment_deposit'] - enriched['subsidy_kaud']).round(2)

print('--- Final Analysis-Ready Admissions Dataset ---')
display(enriched[['applicant_id', 'full_name', 'campus', 'applied_date', 'enrolment_deposit', 'subsidy_rate', 'net_deposit']])

## 5. Practice Exercises

### Exercise: Medical Clinic Invoicing Cleanup
You are given a raw table of patient billing invoices with formatted dollar strings and mixed dates.
1. Clean `billed_amount` to float.
2. Convert `admission_date` into proper datetime objects.
3. Calculate a 10% Medicare rebate column (`rebate_amount`).

In [ ]:
# Exercise Data
dirty_billing = pd.DataFrame({
    'patient_id': ['P-101', 'P-102', 'P-103', 'P-104'],
    'billed_amount': [' $450.50 ', '$1,200.00 AUD', ' 320.00 ', '$2,150.75 AUD'],
    'admission_date': ['2026/03/10', '12-03-2026', '2026.03.15', '2026-03-18']
})

print('Raw Billing Table:')
display(dirty_billing)

# --- Student Solution ---
cleaned_billing = dirty_billing.copy()
cleaned_billing['billed_amount'] = (
    cleaned_billing['billed_amount']
    .str.replace(r'[$,AUD\s]', '', regex=True)
    .astype(float)
)
cleaned_billing['admission_date'] = pd.to_datetime(cleaned_billing['admission_date'], format='mixed')
cleaned_billing['rebate_amount'] = (cleaned_billing['billed_amount'] * 0.10).round(2)
cleaned_billing['out_of_pocket'] = (cleaned_billing['billed_amount'] - cleaned_billing['rebate_amount']).round(2)

print('\nCleaned Billing Invoices:')
display(cleaned_billing)